# Phase 5 — Poisson scoreline model

This self-contained notebook implements and validates the Phase 5 baseline. It uses only pre-match Elo information, an explicitly chronological holdout, separate regularized Poisson regressions for home and away goals, and an `8+` overflow bin that preserves truncated probability mass.

**Configuration:** matches from 2000 onward; first 80% train / final 20% holdout; predictors are scaled pre-match Elo difference and neutral venue; L2 regularization `alpha=0.1`; maximum displayed score `8+`. The intercepts capture the home effect.

In [1]:
from dataclasses import dataclass
from pathlib import Path
from math import exp, factorial
import numpy as np
import pandas as pd
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import log_loss

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'matches_with_elo.csv'
SEED, MAX_SCORE, ALPHA = 20260704, 8, 0.1

@dataclass(frozen=True)
class ScorePrediction:
    expected_home_goals: float
    expected_away_goals: float
    score_matrix: np.ndarray
    home_win: float
    draw: float
    away_win: float
    most_likely_score: tuple[int, int]

def poisson_pmf(k: int, rate: float) -> float:
    if k < 0 or rate < 0 or not np.isfinite(rate):
        raise ValueError('k and rate must be finite and non-negative')
    return exp(-rate) * rate ** k / factorial(k)

def capped_poisson(rate: float, max_score: int = MAX_SCORE) -> np.ndarray:
    if max_score < 1:
        raise ValueError('max_score must be at least 1')
    values = np.array([poisson_pmf(k, rate) for k in range(max_score)], dtype=float)
    return np.append(values, max(0.0, 1.0 - values.sum()))

def score_prediction(home_rate: float, away_rate: float, max_score: int = MAX_SCORE) -> ScorePrediction:
    matrix = np.outer(capped_poisson(home_rate, max_score), capped_poisson(away_rate, max_score))
    matrix /= matrix.sum()
    home_win = float(np.tril(matrix, -1).sum())
    draw = float(np.trace(matrix))
    away_win = float(np.triu(matrix, 1).sum())
    mode = tuple(int(v) for v in np.unravel_index(np.argmax(matrix), matrix.shape))
    return ScorePrediction(home_rate, away_rate, matrix, home_win, draw, away_win, mode)

def design(frame: pd.DataFrame) -> np.ndarray:
    return np.column_stack([frame['elo_difference'].to_numpy(float) / 400.0, frame['neutral'].astype(float)])

matches = pd.read_csv(DATA, parse_dates=['date'])
model_data = matches.loc[matches['date'] >= '2000-01-01'].sort_values(['date', 'match_id']).reset_index(drop=True)
split = int(len(model_data) * 0.8)
train, holdout = model_data.iloc[:split].copy(), model_data.iloc[split:].copy()
assert train['date'].max() <= holdout['date'].min()

home_model = PoissonRegressor(alpha=ALPHA, max_iter=1000).fit(design(train), train['home_score'])
away_model = PoissonRegressor(alpha=ALPHA, max_iter=1000).fit(design(train), train['away_score'])
home_rates = home_model.predict(design(holdout))
away_rates = away_model.predict(design(holdout))

predictions = [score_prediction(h, a) for h, a in zip(home_rates, away_rates)]
poisson_probs = np.array([[p.home_win, p.draw, p.away_win] for p in predictions])
elo_probs = holdout[['elo_home_probability', 'elo_draw_probability', 'elo_away_probability']].to_numpy()
labels = holdout['result'].map({'H': 0, 'D': 1, 'A': 2}).to_numpy()
comparison = pd.DataFrame({
    'model': ['Poisson', 'Elo'],
    'multiclass_log_loss': [log_loss(labels, poisson_probs, labels=[0, 1, 2]), log_loss(labels, elo_probs, labels=[0, 1, 2])],
    'accuracy': [(poisson_probs.argmax(1) == labels).mean(), (elo_probs.argmax(1) == labels).mean()],
})
print(f'Train: {len(train):,} matches, {train.date.min().date()} to {train.date.max().date()}')
print(f'Holdout: {len(holdout):,} matches, {holdout.date.min().date()} to {holdout.date.max().date()}')
print('Home coefficients:', home_model.intercept_, home_model.coef_)
print('Away coefficients:', away_model.intercept_, away_model.coef_)
comparison

C:\Users\moksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\moksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


Train: 20,344 matches, 2000-01-04 to 2021-09-07
Holdout: 5,087 matches, 2021-09-07 to 2026-07-03
Home coefficients: 0.4448063001024255 [ 0.74655662 -0.05760226]
Away coefficients: 0.006868278408556787 [-0.71277525  0.23536118]


,model,multiclass_log_loss,accuracy
0,Poisson,0.889356,0.599568
1,Elo,0.907701,0.599764


In [2]:
# Embedded validation checks: PMF, overflow, aggregation, normalization, leakage, and determinism.
assert np.isclose(poisson_pmf(0, 1.0), np.exp(-1))
assert np.isclose(poisson_pmf(2, 2.0), 2 * np.exp(-2))
marginal = capped_poisson(2.4)
assert len(marginal) == MAX_SCORE + 1 and np.isclose(marginal.sum(), 1.0)
assert marginal[-1] > 0  # probability for 8 or more goals

example = score_prediction(1.7, 1.1)
assert example.score_matrix.shape == (MAX_SCORE + 1, MAX_SCORE + 1)
assert np.isfinite(example.score_matrix).all() and (example.score_matrix >= 0).all()
assert np.isclose(example.score_matrix.sum(), 1.0)
assert np.isclose(example.home_win + example.draw + example.away_win, 1.0)
assert example.home_win > example.away_win
assert np.all(np.isfinite(poisson_probs)) and np.all(poisson_probs >= 0)
assert np.allclose(poisson_probs.sum(axis=1), 1.0)
repeat = np.array([[p.home_win, p.draw, p.away_win] for p in [score_prediction(h, a) for h, a in zip(home_rates[:50], away_rates[:50])]])
assert np.array_equal(repeat, poisson_probs[:50])
assert train.index.max() < holdout.index.min()
print('All Phase 5 validation checks passed.')
print(f'Example 1.7–1.1 xG prediction: H={example.home_win:.3f}, D={example.draw:.3f}, A={example.away_win:.3f}, mode={example.most_likely_score}')

All Phase 5 validation checks passed.
Example 1.7–1.1 xG prediction: H=0.514, D=0.240, A=0.246, mode=(1, 1)


## Interpretation and limitations

The comparison above is fair because both baselines use the identical future-only holdout. The model assumes conditionally independent home and away goal counts and uses the Elo difference as its strength signal. It does not yet model team-specific attack/defence, score dependence, squad availability, or tournament context. The final matrix row and column mean **8 or more**, not exactly eight.